In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
import py3Dmol

## Data Preprocessing and Training Machine Learning Models

- Calculate basic molecular properties of identified target-binding ligands using RDKit
- Additional data preparation (scaling)
- Train machine learning models, e.g.:
    - Linear Regression
    - Ridge Regression
    - Random Forest (RF)
    - Support Vector Machine (SVM)
    - Gradient Boosting Machine (GBM)
    - Feed-forward Neural Network (FFNN)
    - Graph Neural Network (GNN)

### Reading in Data

In [ ]:
chembl_id = "CHEMBL402"

df = pd.read_csv(f"../data/{chembl_id}.csv").drop(columns=["Unnamed: 0"])
df

In [ ]:
# Add mol-type objects to dataframe for calculating molecular properties (see below)

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllColumns = True

# Add mol-type objects without H atoms
df["ROMol"] = df["canonical_smiles"].apply(Chem.MolFromSmiles)
# Add mol-type objects including H atoms
df["ROMolH"] = df["ROMol"].apply(Chem.AddHs)
# Generate 3D coordinates
df["ROMol3D"] = df.apply(lambda row: AllChem.EmbedMolecule(row["ROMolH"], randomSeed=42), axis=1)
df = df[df["ROMol3D"] == 0].reset_index().drop(columns=["index"])
# Optimize the 3D structure using the MMFF94 force field
df["ROMolOPT"] = df.apply(lambda row: AllChem.MMFFOptimizeMolecule(row["ROMolH"], maxIters=2000), axis=1)
df = df[df["ROMolOPT"] == 0].reset_index().drop(columns=["index"])
df

In [ ]:
def display_molecule_3D(mol):
    """
    Display a 3D molecule using py3Dmol.
        Requires a molecule with 3D coordinates.
        mol: RDKit Mol object with 3D coordinates
    """
    mb = Chem.MolToMolBlock(mol)
    p = py3Dmol.view(width=400, height=400)
    p.addModel(mb, 'sdf')
    p.setStyle({'stick':{'colorscheme':'greyCarbon', 'radius': 0.1}, 'sphere':{'scale':0.25}})
    p.setBackgroundColor('0xeeeeee')
    p.zoomTo()
    display(p.show())

In [ ]:
display_molecule_3D(df["ROMolH"][0])

### Calculating Molecular Properties

In [ ]:
# Calculate basic molecular properties of identified target-binding ligands using RDKit

from rdkit.Chem import Descriptors, Descriptors3D

df["MW"] = df["ROMol"].apply(Descriptors.MolWt)
df["logP"] = df["ROMol"].apply(Descriptors.MolLogP)
df["HBA"] = df["ROMol"].apply(Descriptors.NumHAcceptors)
df["HBD"] = df["ROMol"].apply(Descriptors.NumHDonors)

def Ro5(mw, logP, HBA, HBD) -> int:
    Ro5_counter = 0

    if mw <= 500:
      Ro5_counter += 1
    if logP <= 5:
      Ro5_counter += 1
    if HBA <= 10:
      Ro5_counter += 1
    if HBD <= 5:
      Ro5_counter += 1

    return Ro5_counter

df["Ro5"] = df.apply(lambda row: Ro5(row["MW"], row["logP"], row["HBA"], row["HBD"]), axis=1)
df["RB"] = df["ROMol"].apply(Descriptors.NumRotatableBonds)
df["TPSA"] = df["ROMol"].apply(Descriptors.TPSA)
df["ROG"] = df.apply(lambda row: Descriptors3D.RadiusOfGyration(row["ROMolH"]), axis=1)
df

In [ ]:
# Sanity check!
df.describe()

### Splitting into Training, Validation, and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split
trainval_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(trainval_df, test_size=0.25, random_state=42)

train_df = train_df.reset_index().drop(columns=["index"])
val_df = val_df.reset_index().drop(columns=["index"])
test_df = test_df.reset_index().drop(columns=["index"])

In [ ]:
input_cols = ["MW", "logP", "HBA", "HBD", "Ro5", "RB", "TPSA", "ROG"]
target_col = "pIC50"

In [ ]:
train_inputs = train_df[input_cols].copy()
val_inputs = val_df[input_cols].copy()
test_inputs = test_df[input_cols].copy()

train_targets = train_df[target_col].copy()
val_targets = val_df[target_col].copy()
test_targets = test_df[target_col].copy()

### Additional Data Preparation (Scaling)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(df[input_cols])

In [ ]:
train_inputs[input_cols] = scaler.transform(train_inputs[input_cols])
val_inputs[input_cols] = scaler.transform(val_inputs[input_cols])
test_inputs[input_cols] = scaler.transform(test_inputs[input_cols])

In [ ]:
train_inputs.describe().loc[['min', 'max']]

In [ ]:
val_inputs.describe().loc[['min', 'max']]

In [ ]:
test_inputs.describe().loc[['min', 'max']]

### Data Preprocessing Summary

Several dataframes have been created to now hold (scaled) input molecular descriptors and target pIC50 values:
- Input dataframes: `train_inputs`, `val_inputs`, and `test_inputs`
- Target dataframes: `train_targets`, `val_targets`, and `test_targets`

## Baseline Models: Primitive Predictive Models

### Always Predicting the Mean of the Training pIC50 Values

In [ ]:
def mean_guess(inputs, targets):
    return np.full(inputs.shape[0], fill_value=targets.mean())

In [ ]:
train_preds = mean_guess(train_inputs, train_targets)
val_preds = mean_guess(val_inputs, train_targets)        # Use training target range for validation predictions!   
test_preds = mean_guess(test_inputs, train_targets)      # Use training target range for test predictions!

In [ ]:
# Evaluation of "Always Predict the Mean" Model

from sklearn.metrics import root_mean_squared_error

train_rmse = root_mean_squared_error(train_targets, train_preds)
val_rmse = root_mean_squared_error(val_targets, val_preds)
test_rmse = root_mean_squared_error(test_targets, test_preds)

print("Baseline Model Performance (Always Predicting the Mean):")
print(f"Train RMSE (pIC50):         {train_rmse:.4f}")
print(f"Validation RMSE (pIC50):    {val_rmse:.4f}")
print(f"Test RMSE (pIC50):          {test_rmse:.4f}")

### Predicting Random Numbers in the Range of the Training pIC50 Values

In [ ]:
np.random.seed(42)  # For reproducibility

def random_guess(inputs, targets):
    return np.random.uniform(targets.min(), targets.max(), len(inputs))

In [ ]:
train_preds = random_guess(train_inputs, train_targets)
val_preds = random_guess(val_inputs, train_targets)         # Use training target range for validation predictions!
test_preds = random_guess(test_inputs, train_targets)       # Use training target range for test predictions!

In [ ]:
# Evaluation of "Predict Random Numbers" Model

from sklearn.metrics import root_mean_squared_error

train_rmse = root_mean_squared_error(train_targets, train_preds)
val_rmse = root_mean_squared_error(val_targets, val_preds)
test_rmse = root_mean_squared_error(test_targets, test_preds)

print("Baseline Model Performance (Predicting Random Numbers):")
print(f"Train RMSE (pIC50):         {train_rmse:.4f}")
print(f"Validation RMSE (pIC50):    {val_rmse:.4f}")
print(f"Test RMSE (pIC50):          {test_rmse:.4f}")